# Eksplorasi: Apakah `product_return` layak dilanjutkan ke pipeline penuh?

Tujuan notebook ini: cek apakah fitur-fitur seperti riwayat retur historis pengguna, persentase diskon, dan kategori produk memiliki sinyal statistik terhadap probabilitas retur (`is_returned`).

Struktur:
1. Load parquet & cek struktur dasar
2. Distribusi target (`is_returned`)
3. Diagnostik Mutual Information
4. Baseline model cepat (tanpa tuning) -- estimasi kasar PR-AUC
5. Kesimpulan

In [21]:
import pandas as pd
import numpy as np
import glob
import os

from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import LabelEncoder

pd.set_option('display.max_columns', None)

## 1. Load Parquet
Pastikan path ini sesuai dengan struktur lokal Anda.

In [22]:
# Gunakan huruf 'r' di depan tanda kutip agar Windows membaca path dengan benar
PARQUET_DIR = r"D:\PREP_INTERN\PROJECT_DATAFLOW_SIMULATION\datasets\parquet"

PARQUET_FILE = os.path.join(PARQUET_DIR, "ml_product_return_latest.parquet")

df = pd.read_parquet(PARQUET_FILE)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
df.head()

Shape: (30304, 19)

Dtypes:
order_item_id                  int64
order_id                       int64
user_id                      float64
product_id                     int64
is_returned                    int64
product_category                 str
product_department               str
product_brand                    str
retail_price                 float64
sale_price                   float64
discount_amount              float64
discount_percentage          float64
user_age                       int64
user_gender                      str
user_country                     str
traffic_source                   str
historical_items_bought      float64
historical_items_returned    float64
historical_return_rate       float64
dtype: object


,order_item_id,order_id,user_id,product_id,is_returned,product_category,product_department,product_brand,retail_price,sale_price,discount_amount,discount_percentage,user_age,user_gender,user_country,traffic_source,historical_items_bought,historical_items_returned,historical_return_rate
0,73338,50458,40404.0,657,0,Tops & Tees,Women,G by GUESS,34.50,34.50,0.0,0.0,20,F,China,Search,0.0,0.0,0.0
1,138443,95291,76043.0,15262,0,Plus,Women,Shagwear,24.99,24.99,0.0,0.0,69,F,Spain,Search,0.0,0.0,0.0
2,7259,5032,3982.0,8297,1,Outerwear & Coats,Women,Yukon,60.39,60.39,0.0,0.0,53,F,South Korea,Search,0.0,0.0,0.0
3,23014,15930,12735.0,7369,1,Skirts,Women,MAXSTUDIO,98.00,98.00,0.0,0.0,44,F,Brasil,Search,0.0,0.0,0.0
4,116283,80115,63905.0,26153,0,Underwear,Men,HUGO BOSS,18.75,18.75,0.0,0.0,47,M,China,Facebook,0.0,0.0,0.0


## 2. Distribusi Target (`is_returned`)

Mengecek seberapa parah imbalance pada target (berdasarkan kueri, target rate sekitar 11.66%).

In [23]:
target_col = "is_returned"

counts = df[target_col].value_counts()
pct = df[target_col].value_counts(normalize=True) * 100

print("Distribusi target:")
for cls in counts.index:
    print(f"  Kelas {cls}: {counts[cls]:>6d} ({pct[cls]:.2f}%)")

ratio = counts.min() / counts.max()
print(f"\nRasio minoritas:mayoritas = 1:{1/ratio:.1f}")

Distribusi target:
  Kelas 0:  26811 (88.47%)
  Kelas 1:   3493 (11.53%)

Rasio minoritas:mayoritas = 1:7.7


## 3. Diagnostik Mutual Information

Kita akan drop identifier (`order_item_id`, `order_id`, `user_id`, `product_id`) agar tidak bocor atau merusak model[cite: 1].

In [24]:
# 1. Exclude identifier dan target
exclude_cols = ['order_item_id', 'order_id', 'user_id', 'product_id', target_col]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# 2. Pisahkan Numerik & Kategorikal
numeric_features = df[feature_cols].select_dtypes(include=['number']).columns.tolist()
categorical_features = [c for c in feature_cols if c not in numeric_features]

y = df[target_col]

print("=" * 80)
print("MUTUAL INFORMATION vs is_returned")
print("=" * 80)

mi_results = []

if numeric_features:
    X_num = df[numeric_features].fillna(0)
    mi_numeric = mutual_info_classif(X_num, y, random_state=42)
    for feat, score in zip(numeric_features, mi_numeric):
        mi_results.append((feat, 'numeric', score))

if categorical_features:
    for feat in categorical_features:
        le = LabelEncoder()
        encoded = le.fit_transform(df[feat].astype(str).fillna("MISSING"))
        score = mutual_info_score(encoded, y)
        mi_results.append((feat, 'categorical', score))

mi_df = pd.DataFrame(mi_results, columns=['feature', 'type', 'mutual_information'])
mi_df = mi_df.sort_values('mutual_information', ascending=False).reset_index(drop=True)

for _, row in mi_df.iterrows():
    print(f"  [{row['type']:<11s}] {row['feature']:<35s} MI = {row['mutual_information']:.5f}")

print("=" * 80)


MUTUAL INFORMATION vs is_returned


  [categorical] product_brand                       MI = 0.03628
  [numeric    ] discount_percentage                 MI = 0.00215
  [numeric    ] user_age                            MI = 0.00187
  [numeric    ] historical_items_returned           MI = 0.00119
  [numeric    ] historical_return_rate              MI = 0.00088
  [numeric    ] discount_amount                     MI = 0.00074
  [categorical] product_category                    MI = 0.00032
  [categorical] user_country                        MI = 0.00021
  [categorical] traffic_source                      MI = 0.00004
  [categorical] product_department                  MI = 0.00001
  [categorical] user_gender                         MI = 0.00001
  [numeric    ] historical_items_bought             MI = 0.00000
  [numeric    ] retail_price                        MI = 0.00000
  [numeric    ] sale_price                          MI = 0.00000


## 4. Baseline Model Cepat (Tanpa Tuning)

Cek lift dari `baseline_pr_auc`. Jika > 1.5x, kemungkinan besar fitur riwayat retur atau produk memang prediktif.

In [25]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from category_encoders import TargetEncoder
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score, classification_report

X = df[feature_cols]

numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='constant', fill_value=0))])
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('target_encoder', TargetEncoder())
])
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
], remainder='drop')

count_minority = y.value_counts().min()
count_majority = y.value_counts().max()
scale_pos_weight = count_minority / count_majority if y.value_counts().idxmax() == 1 else count_majority / count_minority

# Hati-hati di bagian ini jika data Anda masih terlalu sedikit
n_splits = min(5, max(2, count_minority))

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', XGBClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        scale_pos_weight=scale_pos_weight,
        objective='binary:logistic', tree_method='hist',
        random_state=42, n_jobs=2, eval_metric='logloss'
    ))
])

# Menghasilkan probabilitas dari Cross-Validation
oof_proba = cross_val_predict(
    pipeline, X, y,
    cv=StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42),
    method='predict_proba', n_jobs=1
)[:, 1]

pr_auc = average_precision_score(y, oof_proba)
baseline_pr_auc = y.mean()

print(f"PR-AUC model (out-of-fold, tanpa tuning): {pr_auc:.4f}")
print(f"PR-AUC baseline (tebak proporsi kelas)  : {baseline_pr_auc:.4f}")
print(f"Lift terhadap baseline                  : {pr_auc / baseline_pr_auc:.2f}x")


# ==========================================
# 1. CLASSIFICATION REPORT
# ==========================================
# Mengubah probabilitas menjadi tebakan kelas biner (threshold 0.5)
oof_preds = (oof_proba >= 0.5).astype(int)

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT (Out-of-Fold)")
print("=" * 60)
print(classification_report(y, oof_preds, digits=4))


# ==========================================
# 2. FEATURE IMPORTANCE
# ==========================================
# Wajib melakukan fit ke seluruh data agar tidak NotFittedError
pipeline.fit(X, y)

# Ambil model XGBoost dari dalam Pipeline
xgb_model = pipeline.named_steps['classifier']

# Gabungkan nama fitur (harus sesuai urutan ColumnTransformer)
semua_fitur = numeric_features + categorical_features

# Buat DataFrame untuk visualisasi
importance_df = pd.DataFrame({
    'Feature': semua_fitur,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=False).reset_index(drop=True)

print("\n" + "=" * 50)
print("FEATURE IMPORTANCE (Top 10)")
print("=" * 50)
print(importance_df.head(10))

PR-AUC model (out-of-fold, tanpa tuning): 0.1157
PR-AUC baseline (tebak proporsi kelas)  : 0.1153
Lift terhadap baseline                  : 1.00x

CLASSIFICATION REPORT (Out-of-Fold)
              precision    recall  f1-score   support

           0     0.8872    0.6186    0.7289     26811
           1     0.1192    0.3962    0.1833      3493

    accuracy                         0.5930     30304
   macro avg     0.5032    0.5074    0.4561     30304
weighted avg     0.7987    0.5930    0.6660     30304


FEATURE IMPORTANCE (Top 10)
                     Feature  Importance
0              product_brand    0.334732
1         product_department    0.103961
2           product_category    0.082011
3               retail_price    0.080656
4             traffic_source    0.079186
5               user_country    0.078732
6                   user_age    0.078014
7    historical_items_bought    0.075837
8  historical_items_returned    0.043564
9     historical_return_rate    0.043308


## 5. Kesimpulan

- Apakah fitur `historical_return_rate` mendominasi MI?
- Apakah fitur produk (seperti `product_brand` atau `discount_percentage`) berkontribusi pada lift PR-AUC?
- Tuliskan keputusan Anda: Lanjut ke DAG Optuna atau tidak?